# 3D Gaussian Splatting with GSplat in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varunm18/diffusion-time-machine/blob/varun/notebooks/colab_gsplat_test.ipynb)

This notebook runs the **CUDA-accelerated 3D Gaussian Splatting (`gsplat`)** pipeline using images stored in your Google Drive at `MyDrive/Time-Machine/riverview`.

### Pipeline Stages:
1. **Pose Estimation & Point Cloud**: `mini-DUSt3R` extracts camera intrinsics, camera extrinsics (`transforms.json`), and sparse point cloud (`sparse/0/points3D.ply`).
2. **3DGS Optimization**: `gsplat` trains 3D Gaussians on CUDA GPU using rasterization and adaptive density control.
3. **Export**: Outputs both standard `scene_3dgs.ply` and lightweight `scene_3dgs.splat` directly to your Google Drive.

## 1. Mount Google Drive
Mount your Google Drive to access `MyDrive/Time-Machine/riverview`.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

IMAGE_DIR = Path('/content/drive/MyDrive/Time-Machine/riverview')
assert IMAGE_DIR.exists(), f'Image directory not found at {IMAGE_DIR}. Please check the path in your Google Drive.'

image_files = sorted([f for f in IMAGE_DIR.iterdir() if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Found {len(image_files)} images in {IMAGE_DIR}:')
for img in image_files[:5]:
    print(f'  - {img.name}')
if len(image_files) > 5:
    print(f'  ... and {len(image_files) - 5} more.')

Mounted at /content/drive
Found 12 images in /content/drive/MyDrive/Time-Machine/riverview:
  - 21_DSC0001.jpg
  - 21_DSC0010.jpg
  - 21_DSC0019.jpg
  - 21_DSC0028.jpg
  - 21_DSC0037.jpg
  ... and 7 more.


## 2. Check GPU Hardware Acceleration
Verify that a CUDA-capable GPU (T4, V100, A100, or L4) is assigned to this runtime.
*If GPU is not available, go to **Runtime > Change runtime type > GPU**.*

In [2]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), 'CUDA GPU is required for gsplat. Please enable GPU in Runtime settings.'
print(f'CUDA Device: {torch.cuda.get_device_name(0)}')
print(f'PyTorch Version: {torch.__version__}')

Tue Nov 10 18:58:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Clone Repository & Install Dependencies
Clone the `diffusion-time-machine` repo (branch `varun`) and install `gsplat` alongside project requirements.

In [3]:
import os

# Clone repository if not already cloned
if not os.path.exists('/content/diffusion-time-machine'):
    !git clone -b varun https://github.com/varunm18/diffusion-time-machine.git /content/diffusion-time-machine

%cd /content/diffusion-time-machine
!git pull origin varun

# Install project requirements and CUDA gsplat
!pip install -q -r requirements.txt
!pip install -q gsplat

Cloning into '/content/diffusion-time-machine'...
remote: Enumerating objects: 689, done.
remote: Counting objects: 100% (689/689), done.
remote: Compressing objects: 100% (406/406), done.
remote: Total 689 (delta 338), reused 599 (delta 249), pack-reused 0 (from 0)
Receiving objects: 100% (689/689), 8.75 MiB | 13.72 MiB/s, done.
Resolving deltas: 100% (338/338), done.
/content/diffusion-time-machine
From https://github.com/varunm18/diffusion-time-machine
 * branch            varun      -> FETCH_HEAD
Already up to date.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.


In [5]:
# Verify library imports
import gsplat
from src.viz import DUST3R, GSplat, visualize

print(f'gsplat version: {gsplat.__version__}')
print('src.viz components (DUST3R, GSplat, visualize) loaded successfully!')

gsplat version: 1.5.3
src.viz components (DUST3R, GSplat, visualize) loaded successfully!


## 4. Set Workspace Directory
Configure the destination workspace folder where camera poses, point clouds, and 3DGS models will be saved.
Saving to Google Drive ensures your trained 3D models persist across Colab sessions.

In [6]:
# Store outputs in Google Drive so they persist
WORKSPACE = Path('/content/drive/MyDrive/Time-Machine/riverview_workspace')
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f'Input images : {IMAGE_DIR}')
print(f'Output folder: {WORKSPACE}')

Input images : /content/drive/MyDrive/Time-Machine/riverview
Output folder: /content/drive/MyDrive/Time-Machine/riverview_workspace


## 5. Step 1: mini-DUSt3R Reconstruction
Extracts camera intrinsics, camera extrinsics (`transforms.json`), and sparse point cloud (`sparse/0/points3D.ply`).

In [7]:
# Run mini-DUSt3R camera estimation & initial point cloud on GPU
dust3r = DUST3R(device='cuda')
results = dust3r.run(image_dir=IMAGE_DIR, output_workspace=WORKSPACE)

print(f'Successfully saved camera poses to {WORKSPACE / "transforms.json"}')
print(f'Point cloud contains {len(results.point_cloud.vertices):,} points.')

[DUSt3R] Initializing mini-DUSt3R on device='cuda'


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/450 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.28GB            

model.safetensors: downloading bytes:           |  0.00B            

[DUSt3R] Running mini-DUSt3R inference on 12 images from /content/drive/MyDrive/Time-Machine/riverview...
>> Loading images from /content/drive/MyDrive/Time-Machine/riverview
 - adding 21_DSC0001.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0010.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0019.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0028.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0037.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0046.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0055.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0064.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0073.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0082.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0091.jpg with resolution 739x1113 --> 336x512
 - adding 21_DSC0100.jpg with resolution 739x1113 --> 336x512
 (Found 12 images)
>> Inference with model on 132 image pairs


  0%|          | 0/132 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/mini_dust3r/inference.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/usr/local/lib/python3.13/dist-packages/mini_dust3r/model.py:252: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/lib/python3.13/dist-packages/mini_dust3r/inference.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
100%|██████████| 132/132 [01:23<00:00,  1.58it/s]


 init edge (0*,2*) score=np.float64(32.543701171875)
 init edge (0,1*) score=np.float64(25.824359893798828)
 init edge (0,3*) score=np.float64(25.20371437072754)
 init edge (0,4*) score=np.float64(24.92318344116211)
 init edge (8*,4) score=np.float64(24.752628326416016)
 init edge (7*,4) score=np.float64(26.053390502929688)
 init edge (6*,4) score=np.float64(25.81621551513672)
 init edge (6,9*) score=np.float64(24.71928596496582)
 init edge (10*,7) score=np.float64(23.533222198486328)
 init edge (11*,9) score=np.float64(27.11503791809082)
 init edge (5*,9) score=np.float64(25.0889949798584)
 init loss = 0.01600020006299019
Global alignement - optimizing for:
['pw_poses', 'im_depthmaps', 'im_poses', 'im_focals', 'im_conf.0', 'im_conf.1', 'im_conf.2', 'im_conf.3', 'im_conf.4', 'im_conf.5', 'im_conf.6', 'im_conf.7', 'im_conf.8', 'im_conf.9', 'im_conf.10', 'im_conf.11']


  0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/mini_dust3r/cloud_opt/base_opt.py:390: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  return float(loss)
100%|██████████| 12/12 [00:00<00:00, 16.80it/s]


[DUSt3R] Saved 1,507,562 points to /content/drive/MyDrive/Time-Machine/riverview_workspace/sparse/0/points3D.ply
Saved 12 camera frames to /content/drive/MyDrive/Time-Machine/riverview_workspace/transforms.json
Successfully saved camera poses to /content/drive/MyDrive/Time-Machine/riverview_workspace/transforms.json
Point cloud contains 1,507,562 points.


## 6. Step 2: 3D Gaussian Splatting Training (`gsplat`)
Trains 3D Gaussian Splats on CUDA GPU using `gsplat` rasterization and adaptive density control.

In [9]:
# Set training iterations (7000 is standard for full quality; 2000-3000 for quick testing)
TRAIN_STEPS = 7000

splatter = GSplat(iterations=TRAIN_STEPS, device='cuda')
output_ply = splatter.train(workspace_dir=WORKSPACE, callback_every=500)

print('3DGS training complete!')
print(f'Saved PLY   : {output_ply}')
print(f'Saved Splat : {output_ply.with_suffix(".splat")}')

[gsplat] Loading dataset from /content/drive/MyDrive/Time-Machine/riverview_workspace...
[gsplat] Subsampled initial point cloud from 1,507,562 to 100,000 points
[gsplat] Starting CUDA-accelerated 3DGS optimization (7000 iterations)...
step=0 splats=100,000 loss=0.1342
step=500 splats=100,000 loss=0.0349
step=1000 splats=129,516 loss=0.0279
step=1500 splats=167,122 loss=0.0255
step=2000 splats=198,687 loss=0.0170
step=2500 splats=225,997 loss=0.0153
step=3000 splats=246,345 loss=0.0154
step=3500 splats=264,520 loss=0.0146
step=4000 splats=278,039 loss=0.0162
step=4500 splats=290,079 loss=0.0136
step=5000 splats=300,108 loss=0.0156
step=5500 splats=311,459 loss=0.0148
step=6000 splats=318,030 loss=0.0134
step=6500 splats=318,030 loss=0.0117
[gsplat] Success! 3DGS scene saved to /content/drive/MyDrive/Time-Machine/riverview_workspace/scene_3dgs.ply and /content/drive/MyDrive/Time-Machine/riverview_workspace/scene_3dgs.splat
3DGS training complete!
Saved PLY   : /content/drive/MyDrive/Tim

## 7. Alternative: Single Command CLI Runner
You can also run the entire end-to-end pipeline in one shell command using `scripts/run_3d_pipeline.py`:

In [ ]:
!python scripts/run_3d_pipeline.py \
  --image_dir "/content/drive/MyDrive/Time-Machine/riverview" \
  --workspace "/content/drive/MyDrive/Time-Machine/riverview_workspace" \
  --device cuda \
  --gsplat

## 8. Check Results & Inspect Files
Inspect the exported files. Because `WORKSPACE` is in Google Drive, they are already safely saved to your Google Drive!

In [ ]:
ply_file = WORKSPACE / 'scene_3dgs.ply'
splat_file = WORKSPACE / 'scene_3dgs.splat'
transforms_file = WORKSPACE / 'transforms.json'

for f in [ply_file, splat_file, transforms_file]:
    if f.exists():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'{f.name:<20}: {size_mb:.2f} MB  ({f})')
    else:
        print(f'{f.name:<20}: NOT FOUND')

## 9. How to View Your 3D Gaussian Splat
You have multiple options to view the generated scene:

### Option A: Online Web Splat Viewer (Instant in Browser)
1. Download `scene_3dgs.splat` or locate it in your Google Drive folder.
2. Open [SuperSplat Viewer](https://playcanvas.com/supersplat/editor) or [Antimatter15 Splat Viewer](https://antimatter15.com/splat/).
3. Drag and drop `scene_3dgs.splat` (or `scene_3dgs.ply`) directly into your browser!

### Option B: Local Interactive Rerun Viewer
On your local machine (Mac or PC with repo cloned):
```bash
python -m src.viz.viewer --target /path/to/riverview_workspace
```